In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"D:\Projectsssss\rag-qa-assistant\data\HBO.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print(pages[0].page_content[:500])

C:\Users\SAI VIGNESH CHINTALA\AppData\Local\Temp\ipykernel_34044\1381320381.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 13 pages
HOME BOX
OFFICE
A Comprehensive Retrospective of HBO's Premium
Television Legacy
HBO: A Retrospective
1


#### this code below is a test where our code is actually looking like where it is searching the pdf


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)

print(f"Created {len(chunks)} chunks from {len(pages)} pages")
print(chunks[0].page_content)

Created 28 chunks from 13 pages
HOME BOX
OFFICE
A Comprehensive Retrospective of HBO's Premium
Television Legacy
HBO: A Retrospective
1


##### checking whether chunking acctually worked 

In [3]:
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content)
    print()

--- Chunk 0 ---
HOME BOX
OFFICE
A Comprehensive Retrospective of HBO's Premium
Television Legacy
HBO: A Retrospective
1

--- Chunk 1 ---
Contents
1. The Genesis of HBO: Pioneering Premium Cable
2. The World of Westeros: Game of Thrones
3. The Targaryen Dynasty: House of the Dragon
4. The Smallfolk and the Knights: A Knight of the Seven
Kingdoms
5. Defining the Golden Age: The Sopranos & The Wire
6. Modern Masterpieces: Succession & The Last of Us
7. The Enduring Legacy of HBO
HBO: A Retrospective
2

--- Chunk 2 ---
1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally  known  as  HBO,  pioneered  the  modern  pay  television
landscape. Launched on November 8, 1972, HBO emerged as the first television service
to  be  directly  transmitted  and  distributed  to  individual  cable  television  systems.
Conceptualized by cable executive Charles Dolan as "The Green Channel," the service
was  intended  to  offer  unedited  theatrical  movies  and  live  sporting  ev

#### generate embedddings + store in chromaDB

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

print("Embeddings created and stored successfully!")
print(f"Total chunks stored: {vectorstore._collection.count()}")

Embeddings created and stored successfully!
Total chunks stored: 28


###### SEMANTIC SEARCh


In [5]:
question = "When did HBO launch?"
results = vectorstore.similarity_search(question, k=2)

for i, result in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(result.page_content)
    print()

--- Result 1 ---
1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally  known  as  HBO,  pioneered  the  modern  pay  television
landscape. Launched on November 8, 1972, HBO emerged as the first television service
to  be  directly  transmitted  and  distributed  to  individual  cable  television  systems.
Conceptualized by cable executive Charles Dolan as "The Green Channel," the service
was  intended  to  offer  unedited  theatrical  movies  and  live  sporting  events  without
advertising interruptions, available for a flat monthly fee.
HBO's inaugural broadcast was a National Hockey League (NHL) game between the New
York Rangers and the Vancouver Canucks from Madison Square Garden. This broadcast

--- Result 2 ---
content and reduce reliance on Hollywood studio schedules, the network ventured into
original programming. In 1983, HBO Premiere Films was formed to produce made-for-
cable movies and miniseries with exceptionally high production values. This mark

In [6]:
pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Build keyword search

In [7]:
from rank_bm25 import BM25Okapi

# BM25 needs plain word lists, not raw text - so we tokenize each chunk
tokenized_chunks = [chunk.page_content.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def keyword_search(query, k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    scored_indices = [(i, s) for i, s in enumerate(scores) if s > 0]
    scored_indices.sort(key=lambda x: x[1], reverse=True)
    top_indices = [i for i, s in scored_indices[:k]]
    return [chunks[i] for i in top_indices]

# Quick test
results = keyword_search("Charles Dolan", k=3)
for i, r in enumerate(results):
    print(f"--- Keyword Result {i+1} ---")
    print(r.page_content[:200])
    print()

--- Keyword Result 1 ---
1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally  known  as  HBO,  pioneered  the  modern  pay  television
landscape. Launched on November 8, 1972, HBO emerged as the fi

--- Keyword Result 2 ---
long-form narratives that require patience and intellectual engagement.
The expansion of the Westerosi universe through House of the Dragon and A Knight of the
Seven Kingdoms indicates that HBO remain



#### hybrid search + RRF(reciprocal rank fusion ) fusion code 

In [8]:
def hybrid_search(query, k=3):
    # Get top candidates from both methods (get more than we need, so fusion has options to work with)
    semantic_results = vectorstore.similarity_search(query, k=10)
    keyword_results = keyword_search(query, k=10)

    # Build rank dictionaries: {chunk_text: rank_position}
    semantic_ranks = {doc.page_content: rank for rank, doc in enumerate(semantic_results)}
    keyword_ranks = {doc.page_content: rank for rank, doc in enumerate(keyword_results)}

    # Collect every unique chunk that appeared in EITHER list
    all_chunks = set(semantic_ranks.keys()) | set(keyword_ranks.keys())

    # Apply RRF formula to each chunk
    rrf_scores = {}
    for chunk_text in all_chunks:
        score = 0
        if chunk_text in semantic_ranks:
            score += 1 / (60 + semantic_ranks[chunk_text])
        if chunk_text in keyword_ranks:
            score += 1 / (60 + keyword_ranks[chunk_text])
        rrf_scores[chunk_text] = score

    # Sort by RRF score, highest first, return top k
    sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_chunks[:k]

# Test with the exact-term query
results = hybrid_search("Charles Dolan", k=3)
for i, (text, score) in enumerate(results):
    print(f"--- Hybrid Result {i+1} (RRF score: {score:.5f}) ---")
    print(text[:200])
    print()

--- Hybrid Result 1 (RRF score: 0.03306) ---
long-form narratives that require patience and intellectual engagement.
The expansion of the Westerosi universe through House of the Dragon and A Knight of the
Seven Kingdoms indicates that HBO remain

--- Hybrid Result 2 (RRF score: 0.03306) ---
1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally  known  as  HBO,  pioneered  the  modern  pay  television
landscape. Launched on November 8, 1972, HBO emerged as the fi

--- Hybrid Result 3 (RRF score: 0.01613) ---
flawed, often monstrous protagonist whom audiences nonetheless understood and rooted
for. It elevated TV writing to literary standards, utilizing dream sequences, ambiguous
endings, and stark moral qu



##### debug cell

In [9]:
print("--- Semantic ranks ---")
semantic_results = vectorstore.similarity_search("Charles Dolan", k=10)
for rank, doc in enumerate(semantic_results):
    print(f"{rank}: {doc.page_content[:80]}")

print("\n--- Keyword ranks ---")
keyword_results = keyword_search("Charles Dolan", k=10)
for rank, doc in enumerate(keyword_results):
    print(f"{rank}: {doc.page_content[:80]}")

--- Semantic ranks ---
0: long-form narratives that require patience and intellectual engagement.
The expa
1: 1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally 
2: flawed, often monstrous protagonist whom audiences nonetheless understood and ro
3: Strong), Roman (Kieran Culkin), Shiv (Sarah Snook), and Connor (Alan Ruck)—begin
4: The Sopranos (1999–2007)
Created by David Chase,  The Sopranos is a psychologica
5: 6. Modern Masterpieces: Succession &
The Last of Us
In the post-Game of Thrones 
6: unfathomable  losses  while  Queen  Alicent  grapples  with  the  consequences  
7: Principal Cast
Actor Character House/Affiliation
Sean Bean Eddard "Ned" StarkHou
8: (Stringer Bell), and Michael K. Williams (Omar Little), the series avoided neat 
9: The first season, helmed by showrunner Ira Parker and director Owen Harris, adap

--- Keyword ranks ---
0: 1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally 
1: long-form narratives that r

In [10]:
query = "Charles Dolan"
tokenized_query = query.lower().split()
scores = bm25.get_scores(tokenized_query)

# Find the House of the Dragon chunk specifically and print its real score
for i, chunk in enumerate(chunks):
    if "House of the Dragon" in chunk.page_content:
        print(f"Chunk {i} score for 'Charles Dolan': {scores[i]:.6f}")
        print(f"Contains 'charles': {'charles' in chunk.page_content.lower()}")
        print(f"Contains 'dolan': {'dolan' in chunk.page_content.lower()}")
        print(chunk.page_content[:300])
        print("---")

Chunk 1 score for 'Charles Dolan': 0.000000
Contains 'charles': False
Contains 'dolan': False
Contents
1. The Genesis of HBO: Pioneering Premium Cable
2. The World of Westeros: Game of Thrones
3. The Targaryen Dynasty: House of the Dragon
4. The Smallfolk and the Knights: A Knight of the Seven
Kingdoms
5. Defining the Golden Age: The Sopranos & The Wire
6. Modern Masterpieces: Succession & T
---
Chunk 11 score for 'Charles Dolan': 0.000000
Contains 'charles': False
Contains 'dolan': False
3. The Targaryen Dynasty: House of
the Dragon
Format: Television Series
Seasons: 3 (Ongoing, Premiered August 2022)
Genre: High Fantasy / Drama 
Following the massive success of Game of Thrones, HBO expanded the universe with its
first prequel series,  House of the Dragon. Created by Ryan Condal and
---
Chunk 17 score for 'Charles Dolan': 0.000000
Contains 'charles': False
Contains 'dolan': False
The first season, helmed by showrunner Ira Parker and director Owen Harris, adapts the
first novella, "The

In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [12]:
def rerank(query, candidates, top_n=3):
    pairs = [[query, chunk_text] for chunk_text, _ in candidates]
    scores = reranker.predict(pairs)
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(chunk_text, score) for (chunk_text, _), score in reranked[:top_n]]

# Test with the query that gave us the tie
query = "Charles Dolan"
hybrid_candidates = hybrid_search(query, k=5)
final_results = rerank(query, hybrid_candidates, top_n=3)

for i, (text, score) in enumerate(final_results):
    print(f"--- Reranked Result {i+1} (cross-encoder score: {score:.4f}) ---")
    print(text[:250])
    print()

--- Reranked Result 1 (cross-encoder score: 3.5364) ---
1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally  known  as  HBO,  pioneered  the  modern  pay  television
landscape. Launched on November 8, 1972, HBO emerged as the first television service
to  be  directly  transmitt

--- Reranked Result 2 (cross-encoder score: 3.1457) ---
long-form narratives that require patience and intellectual engagement.
The expansion of the Westerosi universe through House of the Dragon and A Knight of the
Seven Kingdoms indicates that HBO remains committed to massive, immersive world-
building.

--- Reranked Result 3 (cross-encoder score: -9.1158) ---
The Sopranos (1999–2007)
Created by David Chase,  The Sopranos is a psychological dissection of the American
mobster. The series spans 6 seasons and centers on Tony Soprano (played masterfully by
James Gandolfini), a New Jersey mafia boss who suffers



In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1", temperature=0)

In [16]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context provided below. 
If the context does not contain enough information to answer the question, say "I don't have enough information in the document to answer that" — do not use any outside knowledge.

Context:
{context}

Question: {question}

Answer:
""")

In [17]:
def answer_question(question, k_hybrid=5, k_final=3):
    hybrid_candidates = hybrid_search(question, k=k_hybrid)
    reranked = rerank(question, hybrid_candidates, top_n=k_final)
    
    context = "\n\n---\n\n".join([text for text, score in reranked])
    
    prompt = rag_prompt.format(context=context, question=question)
    response = llm.invoke(prompt)
    
    return response.content, reranked

# Test it
answer, sources = answer_question("When did HBO launch?")
print("ANSWER:")
print(answer)
print("\n--- SOURCES USED ---")
for i, (text, score) in enumerate(sources):
    print(f"Source {i+1} (score: {score:.4f}): {text[:150]}...")

ANSWER:
November 8, 1972.

--- SOURCES USED ---
Source 1 (score: 8.5742): 1. The Genesis of HBO: Pioneering
Premium Cable
Home  Box  Office,  universally  known  as  HBO,  pioneered  the  modern  pay  television
landscape. L...
Source 2 (score: 5.4912): content and reduce reliance on Hollywood studio schedules, the network ventured into
original programming. In 1983, HBO Premiere Films was formed to p...
Source 3 (score: 5.4320): York Rangers and the Vancouver Canucks from Madison Square Garden. This broadcast
was  beamed  to  a  modest  subscriber  base  of  365  households  i...


### testing whether the model answers from the document not on its own memory

In [18]:
answer, sources = answer_question("What is the capital of France?")
print("ANSWER:")
print(answer)
print("\n--- SOURCES USED ---")
for i, (text, score) in enumerate(sources):
    print(f"Source {i+1} (score: {score:.4f}): {text[:150]}...")

ANSWER:
I don't have enough information in the document to answer that.

--- SOURCES USED ---
Source 1 (score: -11.0821): 3. The Targaryen Dynasty: House of
the Dragon
Format: Television Series
Seasons: 3 (Ongoing, Premiered August 2022)
Genre: High Fantasy / Drama 
Follo...
Source 2 (score: -11.0907): Strong), Roman (Kieran Culkin), Shiv (Sarah Snook), and Connor (Alan Ruck)—begin a
vicious, unending battle for control of the company.
The series is ...
Source 3 (score: -11.0981): 2. The World of Westeros: Game of
Thrones
Format: Television Series
Seasons: 8 (April 2011 – May 2019)
Genre: High Fantasy / Drama 
Based on George R....
